In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../data/predictive_maintenance.csv")

# Drop columns that would cause data leakage or are not useful
df = df.drop(columns=["UDI", "Product ID", "TWF", "HDF", "PWF", "OSF", "RNF"])

# Encode Type column (L, M, H) into numbers
df["Type"] = df["Type"].map({"L": 0, "M": 1, "H": 2})

X = df.drop("Machine failure", axis=1)
y = df["Machine failure"]

print(X.columns.tolist())
print(X.shape)

['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
(10000, 6)


In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set: {X_test.shape[0]} rows")
print(f"Failures in training: {y_train.sum()}")
print(f"Failures in test: {y_test.sum()}")

Training set: 8000 rows
Test set: 2000 rows
Failures in training: 271
Failures in test: 68


In [3]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

model = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1932
           1       0.88      0.56      0.68        68

    accuracy                           0.98      2000
   macro avg       0.93      0.78      0.84      2000
weighted avg       0.98      0.98      0.98      2000



In [5]:
import numpy as np

# Get probability scores instead of hard predictions
y_proba = model.predict_proba(X_test_scaled)[:, 1]

# Lower threshold from 0.5 to 0.3
y_pred_adjusted = (y_proba >= 0.3).astype(int)

print(classification_report(y_test, y_pred_adjusted))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1932
           1       0.74      0.72      0.73        68

    accuracy                           0.98      2000
   macro avg       0.87      0.86      0.86      2000
weighted avg       0.98      0.98      0.98      2000

